# Database Layer

## 1. Introduction

This notebook implements the database and processing layers for the multimodal archive retrieval system.

The objective of this notebook is to address the problem of ingesting, processing, indexing, and retrieving multimodal archival content while ensuring data privacy and security. The implementation focuses on:

- Multimodal document ingestion
- Metadata normalization
- Embedding generation
- Vector database indexing
- Knowledge graph construction
- Sensitive data detection and masking
- Secure document retrieval

The system is designed to support large-scale digital archives commonly found in corporations, universities, media agencies, and enterprise knowledge management systems.

To simulate a realistic multimodal archival environment, the following datasets are selected:

| Dataset | Purpose |
|---|---|
| MediaSum | Transcript and dialogue archive |
| CNN/DailyMail | News and article archive |
| MSR-VTT | Video and multimedia archive |
| DocVQA | OCR-scanned document archive |

These datasets provide heterogeneous content types including long-form text, OCR documents, transcripts, and multimedia metadata, allowing the system to evaluate retrieval and security mechanisms under realistic archival conditions.

### TODO
- Full multimodal embedding fusion
- Video feature extraction
- Image embedding support
- Query-aware masking policies
- Access-control-aware retrieval
- Hybrid retrieval optimization
- Large-scale evaluation framework

### 1.1. Install and Import Libraries

In [1]:
# !pip install -q python-dotenv qdrant-client neo4j datasets huggingface_hub tqdm pandas numpy matplotlib pillow
# !pip install -q pytesseract
# !pip install -q -U FlagEmbedding

In [2]:
# ============================================
# 1. Standard Library
# ============================================
import base64
import json
import os
import platform
import random
import shutil
import time
from dataclasses import asdict
from itertools import islice
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

# ============================================
# 2. Third-Party Libraries
# ============================================

# --- Data Handling & Analysis ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

# --- Image Processing / OCR ---
from PIL import Image

try:
    import pytesseract
except ImportError:
    pytesseract = None
    print("pytesseract is not available. OCR extraction will be implemented later.")

# --- Vector Database (Qdrant) ---
from qdrant_client import QdrantClient
from qdrant_client.http import models

# --- Graph Database (Neo4j) ---
from neo4j import GraphDatabase

# --- Authentication / Environment ---
from dotenv import load_dotenv
from huggingface_hub import login

# ============================================
# 3. Project Modules
# ============================================
from src.bge_m3_embedding import (
    BGE_M3_DENSE_VECTOR_SIZE,
    BGE_M3_MODEL_NAME,
    BGEM3Embedder,
    batched,
    build_qdrant_points,
    qdrant_point_ids,
)

from src.archive_schema import (
    AccessLevel,
    ArchiveChunk,
    ArchiveDocument,
    DatasetName,
    Modality,
    SensitiveEntity,
    SensitivityLevel,
    apply_security_masking_placeholder,
    archive_chunk_to_qdrant_payload,
    build_embedding_text,
    chunk_text_by_words,
    normalize_text,
    stable_id,
)

# ============================================
# 4. Load Environment Variables
# ============================================
load_dotenv()

True

### 1.2. Environment Setup & Service Initialization

In [3]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

In [4]:
CSV_CACHE_DIR = Path("data") / "csv_cache"
CSV_JSON_COLUMNS = ["source_metadata", "sensitive_entities"]


def _json_serialize_cell(value: Any) -> Any:
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, ensure_ascii=False)
    return value


def _json_parse_cell(value: Any) -> Any:
    if pd.isna(value) or value == "":
        return None
    if not isinstance(value, str):
        return value
    try:
        return json.loads(value)
    except json.JSONDecodeError:
        return value


def dataframe_to_csv(
    df: pd.DataFrame,
    csv_path: os.PathLike[str] | str,
    json_columns: Optional[List[str]] = None,
    index: bool = False,
) -> Path:
    """Export a dataframe to CSV, JSON-encoding nested dict/list columns first."""
    path = Path(csv_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    export_df = df.copy()
    for column in json_columns or []:
        if column in export_df.columns:
            export_df[column] = export_df[column].map(_json_serialize_cell)

    export_df.to_csv(path, index=index)
    print(f"Saved {len(export_df):,} rows to {path}")
    return path


def csv_to_dataframe(
    csv_path: os.PathLike[str] | str,
    json_columns: Optional[List[str]] = None,
    **read_csv_kwargs: Any,
) -> pd.DataFrame:
    """Read a CSV into a dataframe, decoding JSON columns created by dataframe_to_csv."""
    df = pd.read_csv(csv_path, **read_csv_kwargs)

    for column in json_columns or []:
        if column in df.columns:
            df[column] = df[column].map(_json_parse_cell)

    return df


def archive_documents_to_dataframe(documents: List[ArchiveDocument]) -> pd.DataFrame:
    return pd.DataFrame([asdict(document) for document in documents])


def archive_chunks_to_dataframe(chunks: List[ArchiveChunk]) -> pd.DataFrame:
    return pd.DataFrame([archive_chunk_to_qdrant_payload(chunk) for chunk in chunks])


def dataset_cache_paths(dataset_slug: str) -> Tuple[Path, Path]:
    safe_slug = dataset_slug.lower().replace("/", "_").replace(" ", "_")
    return (
        CSV_CACHE_DIR / f"{safe_slug}_documents.csv",
        CSV_CACHE_DIR / f"{safe_slug}_chunks.csv",
    )


#### 1.2.1. Google Colab Setup

Run this section if you are running this notebook on Google Colab. Otherwise, run section 1.3.2.

In [5]:
# from google.colab import userdata

In [6]:
# QDRANT_CLUSTER_URL = userdata.get("QDRANT_CLUSTER_URL")
# QDRANT_CLUSTER_API_KEY = userdata.get("QDRANT_CLUSTER_API_KEY")
# COLLECTION_NAME = "archive-chunks"

In [7]:
# qdrant_client = QdrantClient(
#     url=QDRANT_CLUSTER_URL,
#     api_key=QDRANT_CLUSTER_API_KEY,
# )

# print(qdrant_client.get_collections())

#### 1.2.2. Local Setup

In [8]:
QDRANT_CLUSTER_URL = os.getenv("QDRANT_CLUSTER_URL")
QDRANT_CLUSTER_API_KEY = os.getenv("QDRANT_CLUSTER_API_KEY")
COLLECTION_NAME = "archive-chunks"

In [9]:
qdrant_client = QdrantClient(
    url=QDRANT_CLUSTER_URL,
    api_key=QDRANT_CLUSTER_API_KEY,
)

print(qdrant_client.get_collections())

collections=[]


### 1.3. Configure Tesseract

Tesseractâ€™s installation path varies depending on the operating system (Windows, macOS, Linux), and the notebook must explicitly point to the correct binary location.

Proper configuration ensures that OCR runs reliably across different environments before we begin extracting text for RAG preprocessing.

In [10]:
def configure_tesseract(required: bool = False) -> bool:
    """Configure Tesseract when available; return whether OCR is ready."""
    if pytesseract is None:
        print("pytesseract is not installed. OCR extraction will be skipped.")
        return False

    system = platform.system()
    print(f"Detected OS: {system}")

    possible_paths = []
    install_message = "Install Tesseract and ensure it is available on PATH."

    if system == "Windows":
        possible_paths = [
            r"C:\Program Files\Tesseract-OCR\tesseract.exe",
            r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe",
        ]
        install_message = "Tesseract not found on Windows. Install from: https://github.com/UB-Mannheim/tesseract/wiki"
    elif system == "Linux":
        install_message = "Tesseract not found on Linux. Install with: sudo apt install tesseract-ocr"
    elif system == "Darwin":
        possible_paths = [
            "/usr/local/bin/tesseract",
            "/opt/homebrew/bin/tesseract",
        ]
        install_message = "Install Tesseract on macOS using: brew install tesseract"
    else:
        message = f"Unsupported OS for automatic Tesseract configuration: {system}"
        if required:
            raise OSError(message)
        print(message)
        return False

    for candidate in possible_paths:
        if os.path.exists(candidate):
            pytesseract.pytesseract.tesseract_cmd = candidate
            print(f"Using Tesseract at: {candidate}")
            return True

    found = shutil.which("tesseract")
    if found:
        pytesseract.pytesseract.tesseract_cmd = found
        print(f"Using Tesseract from PATH: {found}")
        return True

    if required:
        raise FileNotFoundError(install_message)

    print(f"{install_message} OCR extraction will be skipped for now.")
    return False

In [11]:
TESSERACT_READY = configure_tesseract()

if TESSERACT_READY and pytesseract is not None:
    print(f"Tesseract OCR ready: {pytesseract.get_tesseract_version()}")
else:
    print("Tesseract OCR is not ready; DocVQA will use the OCR-unavailable fallback text.")

Detected OS: Windows
Using Tesseract at: C:\Program Files\Tesseract-OCR\tesseract.exe
Tesseract OCR ready: 5.5.0.20241111


## 2. Data Preparation

### 2.1. MediaSum

MediaSum is used as the transcript/dialogue archive. The original Hugging Face dataset is `ccdv/mediasum`, but that repository depends on a legacy dataset loading script (`mediasum.py`). Current versions of `datasets` no longer execute remote dataset scripts, so this notebook loads the auto-converted parquet files from `nbroad/mediasum` instead.

The loader uses explicit pinned parquet file paths with the generic `parquet` builder. This avoids both the original dataset script and any script metadata in the mirror repository. The mirror exposes raw transcript turns as `utt` plus speaker labels as `speaker`, so the adapter below reconstructs a transcript-style `raw_text` field while preserving source metadata such as program, date, URL, and title.

By default, this notebook loads the full selected split. For quick tests, set `MEDIASUM_SAMPLE_SIZE` to a small integer.

In [12]:
MEDIASUM_CANONICAL_DATASET_NAME = "ccdv/mediasum"
MEDIASUM_DATASET_NAME = "nbroad/mediasum"
MEDIASUM_PARQUET_REVISION = "b56400644251d1aca3ce857dc6fb7e28fde60bc5"
MEDIASUM_PARQUET_BASE_URI = f"hf://datasets/{MEDIASUM_DATASET_NAME}@{MEDIASUM_PARQUET_REVISION}/mediasum"
MEDIASUM_SPLIT = "train"
MEDIASUM_SAMPLE_SIZE: Optional[int] = None  # Set to a small number for quick tests.
USE_STREAMING = True


def mediasum_parquet_data_files(split: str = MEDIASUM_SPLIT) -> Dict[str, Any]:
    """Return explicit parquet files so `datasets` never tries to execute mediasum.py."""
    if split == "train":
        return {
            split: [
                f"{MEDIASUM_PARQUET_BASE_URI}/mediasum-train-{index:05d}-of-00009.parquet"
                for index in range(9)
            ]
        }

    if split in {"validation", "test"}:
        return {split: f"{MEDIASUM_PARQUET_BASE_URI}/mediasum-{split}.parquet"}

    raise ValueError(f"Unsupported MediaSum split: {split}")


def load_mediasum_sample(
    sample_size: Optional[int] = MEDIASUM_SAMPLE_SIZE,
    split: str = MEDIASUM_SPLIT,
    streaming: bool = USE_STREAMING,
) -> pd.DataFrame:
    """Download/read MediaSum records from script-free parquet files.

    Set sample_size=None to load the full selected split.
    """
    data_files = mediasum_parquet_data_files(split)

    if streaming:
        dataset_iter = load_dataset(
            "parquet",
            data_files=data_files,
            split=split,
            streaming=True,
        )
        records = list(dataset_iter if sample_size is None else islice(dataset_iter, sample_size))
    else:
        split_expr = split if sample_size is None else f"{split}[:{sample_size}]"
        dataset = load_dataset(
            "parquet",
            data_files=data_files,
            split=split_expr,
        )
        records = list(dataset)

    return pd.DataFrame(records)


mediasum_df = load_mediasum_sample()
print(f"Loaded MediaSum records: {len(mediasum_df):,}")
print(f"Columns: {list(mediasum_df.columns)}")
mediasum_df.head()

Loaded MediaSum records: 443,596
Columns: ['id', 'program', 'date', 'url', 'title', 'summary', 'utt', 'speaker']


,id,program,date,url,title,summary,utt,speaker
0,NPR-1,News & Notes,2007-11-28,https://www.npr.org/templates/story/story.php?...,Black Actors Give Bible Star Appeal,"More than 400 black actors, artists and minist...","[Now, moving on, Forest Whitaker as Moses, Tis...","[FARAI CHIDEYA, host, FARAI CHIDEYA, host, Mr...."
1,NPR-2,Weekend Edition Sunday,2016-10-23,https://www.npr.org/2016/10/23/499042298/young...,"Young, First-Time Voters Share Views On Electi...",NPR's Rachel Martin speaks with young voters w...,[You have heard it again and again - this is a...,"[RACHEL MARTIN, HOST, ASHANTI MARTINEZ, LAUREN..."
2,NPR-3,News & Notes,2007-11-30,https://www.npr.org/templates/story/story.php?...,Snapshots: On Solid Ground,"In this week's snapshot, actor and playwright ...","[I came close to running out of luck, when I a...","[Mr. JEFF OBAFEMI CARR (Actor, Playwright), CH..."
3,NPR-4,News & Notes,2007-11-30,https://www.npr.org/templates/story/story.php?...,"Washington, D.C. Facing HIV/AIDS Epidemic",A new study says one in 50 people in the natio...,"[This is NEWS & NOTES. I'm Farai Chideya., In ...","[FARAI CHIDEYA, host, FARAI CHIDEYA, host, Dr...."
4,NPR-5,News & Notes,2007-11-30,https://www.npr.org/templates/story/story.php?...,Coping When AIDS Hits Your Family: Part II,When a family member is diagnosed with HIV/AID...,"[I'm Farai Chideya and this is NEWS & NOTES., ...","[FARAI CHIDEYA, host, FARAI CHIDEYA, host, FAR..."


In [13]:
def build_mediasum_transcript_text(record: Dict[str, Any]) -> str:
    """Extract transcript text from either the old script format or parquet mirror."""
    document = record.get("document")
    if document:
        return normalize_text(document)

    utterances = record.get("utt") or record.get("utterances") or []
    speakers = record.get("speaker") or record.get("speakers") or []

    if not isinstance(utterances, list):
        return normalize_text(utterances)

    lines = []
    for index, utterance in enumerate(utterances):
        utterance_text = normalize_text(utterance)
        if not utterance_text:
            continue

        speaker = None
        if isinstance(speakers, list) and index < len(speakers):
            speaker = normalize_text(speakers[index])

        if speaker:
            lines.append(f"{speaker}: {utterance_text}")
        else:
            lines.append(utterance_text)

    return normalize_text("\n".join(lines))


def mediasum_record_to_document(record: Dict[str, Any]) -> ArchiveDocument:
    source_id = str(record.get("id", "")).strip()
    raw_text = build_mediasum_transcript_text(record)
    summary = normalize_text(record.get("summary", ""))
    title = normalize_text(record.get("title", ""))

    if not source_id:
        source_id = stable_id(DatasetName.MEDIASUM.value, raw_text[:500], summary[:200])

    document_id = stable_id(DatasetName.MEDIASUM.value, source_id)

    return ArchiveDocument(
        document_id=document_id,
        source_id=source_id,
        dataset=DatasetName.MEDIASUM.value,
        modality=Modality.TRANSCRIPT.value,
        title=title or f"MediaSum Transcript {source_id}",
        raw_text=raw_text,
        summary=summary,
        source_metadata={
            "canonical_hf_dataset": MEDIASUM_CANONICAL_DATASET_NAME,
            "hf_dataset": MEDIASUM_DATASET_NAME,
            "hf_revision": MEDIASUM_PARQUET_REVISION,
            "hf_split": MEDIASUM_SPLIT,
            "program": record.get("program"),
            "date": record.get("date"),
            "url": record.get("url"),
            "title": title or None,
            "original_fields": list(record.keys()),
        },
    )


def document_to_archive_chunks(
    document: ArchiveDocument,
    max_words: int = 350,
    overlap_words: int = 60,
) -> List[ArchiveChunk]:
    raw_chunks = chunk_text_by_words(
        document.raw_text,
        max_words=max_words,
        overlap_words=overlap_words,
    )

    chunks: List[ArchiveChunk] = []

    for chunk_index, raw_chunk in enumerate(raw_chunks):
        masked_text, sensitive_entities, sensitivity_level, access_level = apply_security_masking_placeholder(raw_chunk)
        chunk_id = stable_id(document.document_id, chunk_index, raw_chunk[:200])

        embedding_text = build_embedding_text(
            dataset=document.dataset,
            modality=document.modality,
            title=document.title,
            summary=document.summary,
            masked_text=masked_text,
            source_metadata=document.source_metadata,
        )

        chunks.append(
            ArchiveChunk(
                chunk_id=chunk_id,
                document_id=document.document_id,
                source_id=document.source_id,
                dataset=document.dataset,
                modality=document.modality,
                chunk_index=chunk_index,
                title=document.title,
                raw_text=raw_chunk,
                masked_text=masked_text,
                embedding_text=embedding_text,
                summary=document.summary,
                sensitivity_level=sensitivity_level,
                sensitive_entities=sensitive_entities,
                access_level=access_level,
                source_metadata=document.source_metadata,
            )
        )

    return chunks

In [14]:
mediasum_records = mediasum_df.to_dict(orient="records")
mediasum_documents = [mediasum_record_to_document(record) for record in mediasum_records]

mediasum_chunks: List[ArchiveChunk] = []
for document in tqdm(mediasum_documents, desc="Chunking MediaSum documents"):
    mediasum_chunks.extend(document_to_archive_chunks(document))

mediasum_document_df = archive_documents_to_dataframe(mediasum_documents)
mediasum_chunk_df = archive_chunks_to_dataframe(mediasum_chunks)
mediasum_document_csv_path, mediasum_chunk_csv_path = dataset_cache_paths("mediasum")
dataframe_to_csv(mediasum_document_df, mediasum_document_csv_path, json_columns=["source_metadata"])
dataframe_to_csv(mediasum_chunk_df, mediasum_chunk_csv_path, json_columns=CSV_JSON_COLUMNS)

print(f"Documents: {len(mediasum_documents):,}")
print(f"Chunks: {len(mediasum_chunks):,}")


Chunking MediaSum documents:   0%|          | 0/443596 [00:00<?, ?it/s]

Saved 443,596 rows to data\csv_cache\mediasum_documents.csv
Saved 3,595,567 rows to data\csv_cache\mediasum_chunks.csv
Documents: 443,596
Chunks: 3,595,567


In [ ]:
if "_load_archive_chunks_from_csv" not in globals():
    def _chunk_dataframe_to_archive_chunks(chunk_df: pd.DataFrame) -> List[ArchiveChunk]:
        chunk_df = chunk_df.copy()

        if "chunk_index" in chunk_df.columns:
            chunk_df["chunk_index"] = pd.to_numeric(chunk_df["chunk_index"], errors="coerce").fillna(0).astype(int)

        for column in ["dense_vector_ready", "sparse_vector_ready"]:
            if column in chunk_df.columns:
                chunk_df[column] = chunk_df[column].map(
                    lambda value: str(value).strip().lower() == "true"
                    if isinstance(value, str)
                    else bool(value)
                )

        chunk_df = chunk_df.where(pd.notna(chunk_df), None)
        return [ArchiveChunk(**record) for record in chunk_df.to_dict(orient="records")]


    def _read_chunk_csv_dataframe(
        csv_path: os.PathLike[str] | str,
        max_bytes: Optional[int] = None,
        chunksize: int = 50_000,
    ) -> pd.DataFrame:
        if max_bytes is None:
            return csv_to_dataframe(csv_path, json_columns=CSV_JSON_COLUMNS)

        frames = []
        loaded_bytes = 0

        for chunk_df in pd.read_csv(csv_path, chunksize=chunksize):
            chunk_bytes = int(chunk_df.memory_usage(index=True, deep=True).sum())
            remaining_bytes = max_bytes - loaded_bytes

            if chunk_bytes > remaining_bytes:
                average_row_bytes = max(chunk_bytes / max(len(chunk_df), 1), 1)
                rows_to_keep = max(int(remaining_bytes // average_row_bytes), 0)
                if rows_to_keep > 0:
                    chunk_df = chunk_df.iloc[:rows_to_keep].copy()
                    frames.append(chunk_df)
                    loaded_bytes += int(chunk_df.memory_usage(index=True, deep=True).sum())
                break

            frames.append(chunk_df)
            loaded_bytes += chunk_bytes

            if loaded_bytes >= max_bytes:
                break

        if not frames:
            return pd.DataFrame()

        chunk_df = pd.concat(frames, ignore_index=True)
        for column in CSV_JSON_COLUMNS:
            if column in chunk_df.columns:
                chunk_df[column] = chunk_df[column].map(_json_parse_cell)

        print(f"Loaded approximately {loaded_bytes / (1024 ** 3):.2f} GiB from {csv_path}")
        return chunk_df


    def _load_archive_chunks_from_csv(
        csv_path: os.PathLike[str] | str,
        max_bytes: Optional[int] = None,
    ) -> Tuple[pd.DataFrame, List[ArchiveChunk]]:
        chunk_df = _read_chunk_csv_dataframe(csv_path, max_bytes=max_bytes)
        chunks = _chunk_dataframe_to_archive_chunks(chunk_df)
        print(f"Loaded chunks from CSV: {len(chunks):,}")
        return chunk_df, chunks


CHUNK_CSV_READ_LIMIT_BYTES = 1 * 1024 ** 3

_, mediasum_chunk_csv_path = dataset_cache_paths("mediasum")
mediasum_chunk_df, mediasum_chunks = _load_archive_chunks_from_csv(
    mediasum_chunk_csv_path,
    max_bytes=CHUNK_CSV_READ_LIMIT_BYTES,
)


In [15]:
preview_columns = [
    "chunk_id",
    "document_id",
    "dataset",
    "modality",
    "chunk_index",
    "sensitivity_level",
    "access_level",
    "raw_text",
    "masked_text",
]

mediasum_chunk_df[preview_columns].head()


,chunk_id,document_id,dataset,modality,chunk_index,sensitivity_level,access_level,raw_text,masked_text
0,016ab6e11ff9ec666f56855b,5c7aa6d2035899720d170b77,mediasum,transcript,0,S0,public,"FARAI CHIDEYA, host: Now, moving on, Forest Wh...","FARAI CHIDEYA, host: Now, moving on, Forest Wh..."
1,d29c6ca56395d6633221b335,5c7aa6d2035899720d170b77,mediasum,transcript,1,S0,public,"Mr. KYLE BOWSER (Co-producer, ""The Bible Exper...","Mr. KYLE BOWSER (Co-producer, ""The Bible Exper..."
2,86da10fbe86ac8596bf56dce,5c7aa6d2035899720d170b77,mediasum,transcript,2,S0,public,Ms. WENDY RAQUEL ROBINSON (Actress): (As Angel...,Ms. WENDY RAQUEL ROBINSON (Actress): (As Angel...
3,7995a80c7aac78cd846121b1,5c7aa6d2035899720d170b77,mediasum,transcript,3,S0,public,"FARAI CHIDEYA, host: …and podcasting, listenin...","FARAI CHIDEYA, host: …and podcasting, listenin..."
4,9af0e273553ced784b1bc73b,5c7aa6d2035899720d170b77,mediasum,transcript,4,S0,public,"Mr. KYLE BOWSER (Co-producer, ""The Bible Exper...","Mr. KYLE BOWSER (Co-producer, ""The Bible Exper..."


In [16]:
# Inspect one future embedding input.
# This should use masked_text, not raw_text, once the masking layer is implemented.
print(mediasum_chunks[0].embedding_text[:2_000])

Dataset: mediasum
Modality: transcript
Title: Black Actors Give Bible Star Appeal
Summary: More than 400 black actors, artists and ministers are bringing the Gospel to life in the audio book, The Bible Experience:The Complete Bible. Farai Chideya talks with producer Kyle Bowser and actress Wendy Raquel Robinson, who lends her voice to the project.
Metadata: {"canonical_hf_dataset": "ccdv/mediasum", "hf_dataset": "nbroad/mediasum", "hf_revision": "b56400644251d1aca3ce857dc6fb7e28fde60bc5", "hf_split": "train", "program": "News & Notes", "date": "2007-11-28", "url": "https://www.npr.org/templates/story/story.php?storyId=16697288", "title": "Black Actors Give Bible Star Appeal", "original_fields": ["id", "program", "date", "url", "title", "summary", "utt", "speaker"]}
Content: FARAI CHIDEYA, host: Now, moving on, Forest Whitaker as Moses, Tisha Campbell Martin as Mary Magdalene - well, that's all in "The Bible Experience." A New Testament edition was released in 2006. This edition is bill

### 2.2. CNN/DailyMail

CNN/DailyMail is used as the news/article archive. The Hugging Face dataset `abisee/cnn_dailymail` is parquet-backed and provides three main fields: `id`, `article`, and `highlights`.

By default, this notebook loads the full selected split from version `3.0.0`. Each article is converted into the common `ArchiveDocument` schema, with the article body stored as `raw_text` and the highlights stored as `summary`.

In [12]:
CNN_DAILYMAIL_DATASET_NAME = "abisee/cnn_dailymail"
CNN_DAILYMAIL_CONFIG = "3.0.0"
CNN_DAILYMAIL_SPLIT = "train"
CNN_DAILYMAIL_SAMPLE_SIZE: Optional[int] = None  # Set to a small number for quick tests.
CNN_DAILYMAIL_USE_STREAMING = True


def load_cnn_dailymail_sample(
    sample_size: Optional[int] = CNN_DAILYMAIL_SAMPLE_SIZE,
    split: str = CNN_DAILYMAIL_SPLIT,
    config: str = CNN_DAILYMAIL_CONFIG,
    streaming: bool = CNN_DAILYMAIL_USE_STREAMING,
) -> pd.DataFrame:
    """Download/read CNN/DailyMail records from Hugging Face.

    Set sample_size=None to load the full selected split.
    """
    if streaming:
        dataset_iter = load_dataset(
            CNN_DAILYMAIL_DATASET_NAME,
            config,
            split=split,
            streaming=True,
        )
        records = list(dataset_iter if sample_size is None else islice(dataset_iter, sample_size))
    else:
        split_expr = split if sample_size is None else f"{split}[:{sample_size}]"
        dataset = load_dataset(
            CNN_DAILYMAIL_DATASET_NAME,
            config,
            split=split_expr,
        )
        records = list(dataset)

    return pd.DataFrame(records)


cnn_dailymail_df = load_cnn_dailymail_sample()
print(f"Loaded CNN/DailyMail records: {len(cnn_dailymail_df):,}")
print(f"Columns: {list(cnn_dailymail_df.columns)}")
cnn_dailymail_df.head()

Loaded CNN/DailyMail records: 287,113
Columns: ['article', 'highlights', 'id']


,article,highlights,id
0,"LONDON, England (Reuters) -- Harry Potter star...",Harry Potter star Daniel Radcliffe gets £20M f...,42c027e4ff9730fbb3de84c1af0d2c506e41c3e4
1,Editor's note: In our Behind the Scenes series...,Mentally ill inmates in Miami are housed on th...,ee8871b15c50d0db17b0179a6d2beab35065f1e9
2,"MINNEAPOLIS, Minnesota (CNN) -- Drivers who we...","NEW: ""I thought I was going to die,"" driver sa...",06352019a19ae31e527f37f7571c6dd7f0c5da37
3,WASHINGTON (CNN) -- Doctors removed five small...,"Five small polyps found during procedure; ""non...",24521a2abb2e1f5e34e6824e0f9e56904a2b0e88
4,(CNN) -- The National Football League has ind...,"NEW: NFL chief, Atlanta Falcons owner critical...",7fe70cc8b12fab2d0a258fababf7d9c6b5e1262a


In [13]:
def cnn_dailymail_record_to_document(record: Dict[str, Any]) -> ArchiveDocument:
    source_id = str(record.get("id", "")).strip()
    raw_text = normalize_text(record.get("article", ""))
    summary = normalize_text(record.get("highlights", ""))

    if not source_id:
        source_id = stable_id(DatasetName.CNN_DAILYMAIL.value, raw_text[:500], summary[:200])

    document_id = stable_id(DatasetName.CNN_DAILYMAIL.value, source_id)
    title = f"CNN/DailyMail Article {source_id[:12]}"

    return ArchiveDocument(
        document_id=document_id,
        source_id=source_id,
        dataset=DatasetName.CNN_DAILYMAIL.value,
        modality=Modality.ARTICLE.value,
        title=title,
        raw_text=raw_text,
        summary=summary,
        source_metadata={
            "hf_dataset": CNN_DAILYMAIL_DATASET_NAME,
            "hf_config": CNN_DAILYMAIL_CONFIG,
            "hf_split": CNN_DAILYMAIL_SPLIT,
            "original_fields": list(record.keys()),
        },
    )


def cnn_dailymail_document_to_archive_chunks(
    document: ArchiveDocument,
    max_words: int = 350,
    overlap_words: int = 60,
) -> List[ArchiveChunk]:
    raw_chunks = chunk_text_by_words(
        document.raw_text,
        max_words=max_words,
        overlap_words=overlap_words,
    )

    chunks: List[ArchiveChunk] = []

    for chunk_index, raw_chunk in enumerate(raw_chunks):
        masked_text, sensitive_entities, sensitivity_level, access_level = apply_security_masking_placeholder(raw_chunk)
        chunk_id = stable_id(document.document_id, chunk_index, raw_chunk[:200])

        embedding_text = build_embedding_text(
            dataset=document.dataset,
            modality=document.modality,
            title=document.title,
            summary=document.summary,
            masked_text=masked_text,
            source_metadata=document.source_metadata,
        )

        chunks.append(
            ArchiveChunk(
                chunk_id=chunk_id,
                document_id=document.document_id,
                source_id=document.source_id,
                dataset=document.dataset,
                modality=document.modality,
                chunk_index=chunk_index,
                title=document.title,
                raw_text=raw_chunk,
                masked_text=masked_text,
                embedding_text=embedding_text,
                summary=document.summary,
                sensitivity_level=sensitivity_level,
                sensitive_entities=sensitive_entities,
                access_level=access_level,
                source_metadata=document.source_metadata,
            )
        )

    return chunks

In [14]:
cnn_dailymail_records = cnn_dailymail_df.to_dict(orient="records")
cnn_dailymail_documents = [cnn_dailymail_record_to_document(record) for record in cnn_dailymail_records]

cnn_dailymail_chunks: List[ArchiveChunk] = []
for document in tqdm(cnn_dailymail_documents, desc="Chunking CNN/DailyMail documents"):
    cnn_dailymail_chunks.extend(cnn_dailymail_document_to_archive_chunks(document))

cnn_dailymail_document_df = archive_documents_to_dataframe(cnn_dailymail_documents)
cnn_dailymail_chunk_df = archive_chunks_to_dataframe(cnn_dailymail_chunks)
cnn_dailymail_document_csv_path, cnn_dailymail_chunk_csv_path = dataset_cache_paths("cnn_dailymail")
dataframe_to_csv(cnn_dailymail_document_df, cnn_dailymail_document_csv_path, json_columns=["source_metadata"])
dataframe_to_csv(cnn_dailymail_chunk_df, cnn_dailymail_chunk_csv_path, json_columns=CSV_JSON_COLUMNS)

print(f"Documents: {len(cnn_dailymail_documents):,}")
print(f"Chunks: {len(cnn_dailymail_chunks):,}")


Chunking CNN/DailyMail documents:   0%|          | 0/287113 [00:00<?, ?it/s]

Saved 287,113 rows to data\csv_cache\cnn_dailymail_documents.csv
Saved 930,585 rows to data\csv_cache\cnn_dailymail_chunks.csv
Documents: 287,113
Chunks: 930,585


In [ ]:
if "_load_archive_chunks_from_csv" not in globals():
    def _chunk_dataframe_to_archive_chunks(chunk_df: pd.DataFrame) -> List[ArchiveChunk]:
        chunk_df = chunk_df.copy()

        if "chunk_index" in chunk_df.columns:
            chunk_df["chunk_index"] = pd.to_numeric(chunk_df["chunk_index"], errors="coerce").fillna(0).astype(int)

        for column in ["dense_vector_ready", "sparse_vector_ready"]:
            if column in chunk_df.columns:
                chunk_df[column] = chunk_df[column].map(
                    lambda value: str(value).strip().lower() == "true"
                    if isinstance(value, str)
                    else bool(value)
                )

        chunk_df = chunk_df.where(pd.notna(chunk_df), None)
        return [ArchiveChunk(**record) for record in chunk_df.to_dict(orient="records")]


    def _read_chunk_csv_dataframe(
        csv_path: os.PathLike[str] | str,
        max_bytes: Optional[int] = None,
        chunksize: int = 50_000,
    ) -> pd.DataFrame:
        if max_bytes is None:
            return csv_to_dataframe(csv_path, json_columns=CSV_JSON_COLUMNS)

        frames = []
        loaded_bytes = 0

        for chunk_df in pd.read_csv(csv_path, chunksize=chunksize):
            chunk_bytes = int(chunk_df.memory_usage(index=True, deep=True).sum())
            remaining_bytes = max_bytes - loaded_bytes

            if chunk_bytes > remaining_bytes:
                average_row_bytes = max(chunk_bytes / max(len(chunk_df), 1), 1)
                rows_to_keep = max(int(remaining_bytes // average_row_bytes), 0)
                if rows_to_keep > 0:
                    chunk_df = chunk_df.iloc[:rows_to_keep].copy()
                    frames.append(chunk_df)
                    loaded_bytes += int(chunk_df.memory_usage(index=True, deep=True).sum())
                break

            frames.append(chunk_df)
            loaded_bytes += chunk_bytes

            if loaded_bytes >= max_bytes:
                break

        if not frames:
            return pd.DataFrame()

        chunk_df = pd.concat(frames, ignore_index=True)
        for column in CSV_JSON_COLUMNS:
            if column in chunk_df.columns:
                chunk_df[column] = chunk_df[column].map(_json_parse_cell)

        print(f"Loaded approximately {loaded_bytes / (1024 ** 3):.2f} GiB from {csv_path}")
        return chunk_df


    def _load_archive_chunks_from_csv(
        csv_path: os.PathLike[str] | str,
        max_bytes: Optional[int] = None,
    ) -> Tuple[pd.DataFrame, List[ArchiveChunk]]:
        chunk_df = _read_chunk_csv_dataframe(csv_path, max_bytes=max_bytes)
        chunks = _chunk_dataframe_to_archive_chunks(chunk_df)
        print(f"Loaded chunks from CSV: {len(chunks):,}")
        return chunk_df, chunks


CHUNK_CSV_READ_LIMIT_BYTES = 1 * 1024 ** 3

_, cnn_dailymail_chunk_csv_path = dataset_cache_paths("cnn_dailymail")
cnn_dailymail_chunk_df, cnn_dailymail_chunks = _load_archive_chunks_from_csv(
    cnn_dailymail_chunk_csv_path,
    max_bytes=CHUNK_CSV_READ_LIMIT_BYTES,
)


In [15]:
preview_columns = [
    "chunk_id",
    "document_id",
    "dataset",
    "modality",
    "chunk_index",
    "sensitivity_level",
    "access_level",
    "raw_text",
    "masked_text",
]

cnn_dailymail_chunk_df[preview_columns].head()


,chunk_id,document_id,dataset,modality,chunk_index,sensitivity_level,access_level,raw_text,masked_text
0,5309caf85802afc3e9f6d9aa,2b9c06ea12675e1a5070f338,cnn_dailymail,article,0,S0,public,"LONDON, England (Reuters) -- Harry Potter star...","LONDON, England (Reuters) -- Harry Potter star..."
1,d4f32ce2e055c21da3afd3cd,2b9c06ea12675e1a5070f338,cnn_dailymail,article,1,S0,public,"Despite his growing fame and riches, the actor...","Despite his growing fame and riches, the actor..."
2,25df6a2b3a89c0b5d8cb0df9,84c7300fd150160a8612d8c8,cnn_dailymail,article,0,S0,public,Editor's note: In our Behind the Scenes series...,Editor's note: In our Behind the Scenes series...
3,7dfc63e3766005eaa7e4f91d,84c7300fd150160a8612d8c8,cnn_dailymail,article,1,S0,public,Even though we were not exactly welcomed with ...,Even though we were not exactly welcomed with ...
4,78719743200ba890cca1c4fd,84c7300fd150160a8612d8c8,cnn_dailymail,article,2,S0,public,Leifman tells me that these prisoner-patients ...,Leifman tells me that these prisoner-patients ...


In [16]:
# Inspect one future embedding input.
# This should use masked_text, not raw_text, once the masking layer is implemented.
print(cnn_dailymail_chunks[0].embedding_text[:2_000])

Dataset: cnn_dailymail
Modality: article
Title: CNN/DailyMail Article 42c027e4ff97
Summary: Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .
Metadata: {"hf_dataset": "abisee/cnn_dailymail", "hf_config": "3.0.0", "hf_split": "train", "original_fields": ["article", "highlights", "id"]}
Content: LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him.

Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy th

### 2.3. MSR-VTT

MSR-VTT is used as the video archive in a text-projected form. The Hugging Face dataset `AlexZigma/msr-vtt` is parquet-backed and provides one caption row per video segment, with fields such as `video_id`, `caption`, `sen_id`, `category`, `url`, `start time`, and `end time`.

For the current database layer, we do not ingest native video frames yet. Instead, captions and segment timing metadata are grouped by `video_id` and converted into one text document per video. Each caption becomes a timestamped segment line, allowing `chunk_text_by_words` to preserve caption/segment boundaries during chunking.

TODO:
- Add native video/frame embeddings.
- Add richer category labels.
- Add sensitive data masking.
- Add topic extraction.
- Add dense and sparse embeddings.

In [17]:
MSR_VTT_DATASET_NAME = "AlexZigma/msr-vtt"
MSR_VTT_CONFIG = "default"
MSR_VTT_SPLIT = "train"
MSR_VTT_SAMPLE_SIZE: Optional[int] = None  # Set to a small number for quick tests.
MSR_VTT_USE_STREAMING = True


def load_msr_vtt_sample(
    sample_size: Optional[int] = MSR_VTT_SAMPLE_SIZE,
    split: str = MSR_VTT_SPLIT,
    config: str = MSR_VTT_CONFIG,
    streaming: bool = MSR_VTT_USE_STREAMING,
) -> pd.DataFrame:
    """Download/read MSR-VTT caption rows from Hugging Face.

    Set sample_size=None to load the full selected split.
    """
    if streaming:
        dataset_iter = load_dataset(
            MSR_VTT_DATASET_NAME,
            config,
            split=split,
            streaming=True,
        )
        records = list(dataset_iter if sample_size is None else islice(dataset_iter, sample_size))
    else:
        split_expr = split if sample_size is None else f"{split}[:{sample_size}]"
        dataset = load_dataset(
            MSR_VTT_DATASET_NAME,
            config,
            split=split_expr,
        )
        records = list(dataset)

    return pd.DataFrame(records)


msr_vtt_df = load_msr_vtt_sample()
print(f"Loaded MSR-VTT caption rows: {len(msr_vtt_df):,}")
print(f"Columns: {list(msr_vtt_df.columns)}")
msr_vtt_df.head()

Loaded MSR-VTT caption rows: 6,513
Columns: ['video_id', 'caption', 'sen_id', 'category', 'url', 'start time', 'end time', 'split', 'id', '__index_level_0__']


,video_id,caption,sen_id,category,url,start time,end time,split,id,__index_level_0__
0,video0,a car is shown,77300,9,https://www.youtube.com/watch?v=9lZi22qLlEo,137.72,149.44,train,0,0
1,video1,in a kitchen a woman adds different ingredient...,110460,16,https://www.youtube.com/watch?v=w4JM08PDEng,184.33,206.89,train,1,1
2,video10,a man holds two dogs,47320,6,https://www.youtube.com/watch?v=CcJwo2eyfI0,33.33,46.53,train,10,2
3,video100,a basset hound sits outside a door,18360,12,https://www.youtube.com/watch?v=6S-47swQBBU,1146.06,1156.44,train,100,3
4,video1000,a woman is wearing a costume,49000,7,https://www.youtube.com/watch?v=ALrHNDBK-jw,738.93,749.93,train,1000,4


In [18]:
def format_msr_vtt_time(value: Any) -> str:
    """Format segment times consistently for text-projected video chunks."""
    try:
        return f"{float(value):.2f}s"
    except (TypeError, ValueError):
        return "unknown"


def build_msr_vtt_segment_text(row: Dict[str, Any], segment_index: int) -> str:
    caption = normalize_text(row.get("caption", ""))
    start_time = format_msr_vtt_time(row.get("start time"))
    end_time = format_msr_vtt_time(row.get("end time"))
    return f"Segment {segment_index} [{start_time}-{end_time}]: {caption}"


def msr_vtt_group_to_document(video_id: str, group: pd.DataFrame) -> ArchiveDocument:
    group = group.sort_values(by=["start time", "end time", "sen_id"], na_position="last")
    records = group.to_dict(orient="records")

    segment_lines = [
        build_msr_vtt_segment_text(record, segment_index=index)
        for index, record in enumerate(records)
        if normalize_text(record.get("caption", ""))
    ]
    raw_text = normalize_text("\n".join(segment_lines))

    captions = [normalize_text(record.get("caption", "")) for record in records]
    summary = normalize_text(" ".join(caption for caption in captions[:5] if caption))
    source_id = normalize_text(video_id) or stable_id(DatasetName.MSR_VTT.value, raw_text[:500])
    document_id = stable_id(DatasetName.MSR_VTT.value, source_id)

    first_record = records[0] if records else {}
    category_values = sorted({record.get("category") for record in records if record.get("category") is not None})
    url_values = sorted({record.get("url") for record in records if record.get("url")})

    return ArchiveDocument(
        document_id=document_id,
        source_id=source_id,
        dataset=DatasetName.MSR_VTT.value,
        modality=Modality.VIDEO.value,
        title=f"MSR-VTT Video {source_id}",
        raw_text=raw_text,
        summary=summary,
        source_metadata={
            "hf_dataset": MSR_VTT_DATASET_NAME,
            "hf_config": MSR_VTT_CONFIG,
            "hf_split": MSR_VTT_SPLIT,
            "video_id": source_id,
            "caption_count": len(segment_lines),
            "category_values": category_values,
            "url": url_values[0] if url_values else first_record.get("url"),
            "segment_start_min": group["start time"].min() if "start time" in group else None,
            "segment_end_max": group["end time"].max() if "end time" in group else None,
            "original_fields": list(group.columns),
        },
    )


def msr_vtt_document_to_archive_chunks(
    document: ArchiveDocument,
    max_words: int = 350,
    overlap_words: int = 60,
) -> List[ArchiveChunk]:
    raw_chunks = chunk_text_by_words(
        document.raw_text,
        max_words=max_words,
        overlap_words=overlap_words,
    )

    chunks: List[ArchiveChunk] = []

    for chunk_index, raw_chunk in enumerate(raw_chunks):
        masked_text, sensitive_entities, sensitivity_level, access_level = apply_security_masking_placeholder(raw_chunk)
        chunk_id = stable_id(document.document_id, chunk_index, raw_chunk[:200])

        embedding_text = build_embedding_text(
            dataset=document.dataset,
            modality=document.modality,
            title=document.title,
            summary=document.summary,
            masked_text=masked_text,
            source_metadata=document.source_metadata,
        )

        chunks.append(
            ArchiveChunk(
                chunk_id=chunk_id,
                document_id=document.document_id,
                source_id=document.source_id,
                dataset=document.dataset,
                modality=document.modality,
                chunk_index=chunk_index,
                title=document.title,
                raw_text=raw_chunk,
                masked_text=masked_text,
                embedding_text=embedding_text,
                summary=document.summary,
                sensitivity_level=sensitivity_level,
                sensitive_entities=sensitive_entities,
                access_level=access_level,
                source_metadata=document.source_metadata,
            )
        )

    return chunks

In [19]:
msr_vtt_documents = [
    msr_vtt_group_to_document(str(video_id), group)
    for video_id, group in msr_vtt_df.groupby("video_id", sort=True)
]

msr_vtt_chunks: List[ArchiveChunk] = []
for document in tqdm(msr_vtt_documents, desc="Chunking MSR-VTT video captions"):
    msr_vtt_chunks.extend(msr_vtt_document_to_archive_chunks(document))

msr_vtt_document_df = archive_documents_to_dataframe(msr_vtt_documents)
msr_vtt_chunk_df = archive_chunks_to_dataframe(msr_vtt_chunks)
msr_vtt_document_csv_path, msr_vtt_chunk_csv_path = dataset_cache_paths("msr_vtt")
dataframe_to_csv(msr_vtt_document_df, msr_vtt_document_csv_path, json_columns=["source_metadata"])
dataframe_to_csv(msr_vtt_chunk_df, msr_vtt_chunk_csv_path, json_columns=CSV_JSON_COLUMNS)

print(f"Video documents: {len(msr_vtt_documents):,}")
print(f"Chunks: {len(msr_vtt_chunks):,}")


Chunking MSR-VTT video captions:   0%|          | 0/6513 [00:00<?, ?it/s]

Saved 6,513 rows to data\csv_cache\msr_vtt_documents.csv
Saved 6,513 rows to data\csv_cache\msr_vtt_chunks.csv
Video documents: 6,513
Chunks: 6,513


In [ ]:
if "_load_archive_chunks_from_csv" not in globals():
    def _chunk_dataframe_to_archive_chunks(chunk_df: pd.DataFrame) -> List[ArchiveChunk]:
        chunk_df = chunk_df.copy()

        if "chunk_index" in chunk_df.columns:
            chunk_df["chunk_index"] = pd.to_numeric(chunk_df["chunk_index"], errors="coerce").fillna(0).astype(int)

        for column in ["dense_vector_ready", "sparse_vector_ready"]:
            if column in chunk_df.columns:
                chunk_df[column] = chunk_df[column].map(
                    lambda value: str(value).strip().lower() == "true"
                    if isinstance(value, str)
                    else bool(value)
                )

        chunk_df = chunk_df.where(pd.notna(chunk_df), None)
        return [ArchiveChunk(**record) for record in chunk_df.to_dict(orient="records")]


    def _read_chunk_csv_dataframe(
        csv_path: os.PathLike[str] | str,
        max_bytes: Optional[int] = None,
        chunksize: int = 50_000,
    ) -> pd.DataFrame:
        if max_bytes is None:
            return csv_to_dataframe(csv_path, json_columns=CSV_JSON_COLUMNS)

        frames = []
        loaded_bytes = 0

        for chunk_df in pd.read_csv(csv_path, chunksize=chunksize):
            chunk_bytes = int(chunk_df.memory_usage(index=True, deep=True).sum())
            remaining_bytes = max_bytes - loaded_bytes

            if chunk_bytes > remaining_bytes:
                average_row_bytes = max(chunk_bytes / max(len(chunk_df), 1), 1)
                rows_to_keep = max(int(remaining_bytes // average_row_bytes), 0)
                if rows_to_keep > 0:
                    chunk_df = chunk_df.iloc[:rows_to_keep].copy()
                    frames.append(chunk_df)
                    loaded_bytes += int(chunk_df.memory_usage(index=True, deep=True).sum())
                break

            frames.append(chunk_df)
            loaded_bytes += chunk_bytes

            if loaded_bytes >= max_bytes:
                break

        if not frames:
            return pd.DataFrame()

        chunk_df = pd.concat(frames, ignore_index=True)
        for column in CSV_JSON_COLUMNS:
            if column in chunk_df.columns:
                chunk_df[column] = chunk_df[column].map(_json_parse_cell)

        print(f"Loaded approximately {loaded_bytes / (1024 ** 3):.2f} GiB from {csv_path}")
        return chunk_df


    def _load_archive_chunks_from_csv(
        csv_path: os.PathLike[str] | str,
        max_bytes: Optional[int] = None,
    ) -> Tuple[pd.DataFrame, List[ArchiveChunk]]:
        chunk_df = _read_chunk_csv_dataframe(csv_path, max_bytes=max_bytes)
        chunks = _chunk_dataframe_to_archive_chunks(chunk_df)
        print(f"Loaded chunks from CSV: {len(chunks):,}")
        return chunk_df, chunks


CHUNK_CSV_READ_LIMIT_BYTES = 1 * 1024 ** 3

_, msr_vtt_chunk_csv_path = dataset_cache_paths("msr_vtt")
msr_vtt_chunk_df, msr_vtt_chunks = _load_archive_chunks_from_csv(msr_vtt_chunk_csv_path)


In [20]:
preview_columns = [
    "chunk_id",
    "document_id",
    "dataset",
    "modality",
    "chunk_index",
    "sensitivity_level",
    "access_level",
    "raw_text",
    "masked_text",
]

msr_vtt_chunk_df[preview_columns].head()


,chunk_id,document_id,dataset,modality,chunk_index,sensitivity_level,access_level,raw_text,masked_text
0,64171e284662ec2c6bdc83b0,05350f5bccc6d850ff3dc148,msr_vtt,video,0,S0,public,Segment 0 [137.72s-149.44s]: a car is shown,Segment 0 [137.72s-149.44s]: a car is shown
1,a8504c9e56426adb2b18adaf,8e6c6a2b74aa020aa78eca5e,msr_vtt,video,0,S0,public,Segment 0 [184.33s-206.89s]: in a kitchen a wo...,Segment 0 [184.33s-206.89s]: in a kitchen a wo...
2,13bec3aecd4e69ea1d240415,7f428963ae3522cb089ec22a,msr_vtt,video,0,S0,public,Segment 0 [33.33s-46.53s]: a man holds two dogs,Segment 0 [33.33s-46.53s]: a man holds two dogs
3,f719c4364e1d489517cf0b49,7bef8eb64e628368072a9bf5,msr_vtt,video,0,S0,public,Segment 0 [1146.06s-1156.44s]: a basset hound ...,Segment 0 [1146.06s-1156.44s]: a basset hound ...
4,a567649ba01d75613b60d81e,f3f32bc87278bdc71368de62,msr_vtt,video,0,S0,public,Segment 0 [738.93s-749.93s]: a woman is wearin...,Segment 0 [738.93s-749.93s]: a woman is wearin...


In [21]:
# Inspect one future embedding input.
# This should use masked_text, not raw_text, once the masking layer is implemented.
print(msr_vtt_chunks[0].embedding_text[:2_000])

Dataset: msr_vtt
Modality: video
Title: MSR-VTT Video video0
Summary: a car is shown
Metadata: {"hf_dataset": "AlexZigma/msr-vtt", "hf_config": "default", "hf_split": "train", "video_id": "video0", "caption_count": 1, "category_values": [9], "url": "https://www.youtube.com/watch?v=9lZi22qLlEo", "segment_start_min": 137.72, "segment_end_max": 149.44, "original_fields": ["video_id", "caption", "sen_id", "category", "url", "start time", "end time", "split", "id", "__index_level_0__"]}
Content: Segment 0 [137.72s-149.44s]: a car is shown


### 2.4. DocVQA

DocVQA is used as the OCR-scanned document archive. The Hugging Face dataset `lmms-lab/DocVQA` provides document page images and question-answer annotations with fields such as `questionId`, `question`, `question_types`, `image`, `docId`, `ucsf_document_id`, `ucsf_document_page_no`, `answers`, and `data_split`.

For the database layer, each unique document page is converted into one `ArchiveDocument`. OCR text is extracted from the page image with Tesseract when available, formatted as layout-aware `OCR Line ...` records, and then chunked while preserving those layout lines. Ground-truth answers are intentionally not included in `raw_text`, `masked_text`, or embedding metadata to avoid leaking evaluation labels into retrieval.

TODO:
- Install/configure Tesseract in the runtime environment for full OCR extraction.
- Replace baseline OCR with document-layout OCR or OCR tokens with bounding boxes.
- Add native image/layout embeddings.
- Add sensitive data masking.
- Add topic extraction.
- Add dense and sparse embeddings.

In [22]:
DOCVQA_DATASET_NAME = "lmms-lab/DocVQA"
DOCVQA_CONFIG = "DocVQA"
DOCVQA_SPLIT = "validation"
DOCVQA_SAMPLE_SIZE: Optional[int] = None  # Set to a small number for quick tests.
DOCVQA_USE_STREAMING = True
DOCVQA_ENABLE_OCR = True


def load_docvqa_sample(
    sample_size: Optional[int] = DOCVQA_SAMPLE_SIZE,
    split: str = DOCVQA_SPLIT,
    config: str = DOCVQA_CONFIG,
    streaming: bool = DOCVQA_USE_STREAMING,
) -> pd.DataFrame:
    """Download/read DocVQA rows from Hugging Face.

    Set sample_size=None to load the full selected split.
    """
    if streaming:
        dataset_iter = load_dataset(
            DOCVQA_DATASET_NAME,
            config,
            split=split,
            streaming=True,
        )
        records = list(dataset_iter if sample_size is None else islice(dataset_iter, sample_size))
    else:
        split_expr = split if sample_size is None else f"{split}[:{sample_size}]"
        dataset = load_dataset(
            DOCVQA_DATASET_NAME,
            config,
            split=split_expr,
        )
        records = list(dataset)

    return pd.DataFrame(records)


docvqa_df = load_docvqa_sample()
print(f"Loaded DocVQA question rows: {len(docvqa_df):,}")
print(f"Columns: {list(docvqa_df.columns)}")
docvqa_df.head()

Loaded DocVQA question rows: 5,349
Columns: ['questionId', 'question', 'question_types', 'image', 'docId', 'ucsf_document_id', 'ucsf_document_page_no', 'answers', 'data_split']


,questionId,question,question_types,image,docId,ucsf_document_id,ucsf_document_page_no,answers,data_split
0,49153,"What is the ‘actual’ value per 1000, during th...",[figure/diagram],<PIL.PngImagePlugin.PngImageFile image mode=L ...,14465,pybv0228,81,[0.28],val
1,24580,What is name of university?,[others],<PIL.PngImagePlugin.PngImageFile image mode=L ...,7027,nkbl0226,1,"[university of california, University of Calif...",val
2,57349,What is the name of the company?,[layout],<PIL.PngImagePlugin.PngImageFile image mode=L ...,4733,snbx0223,22,"[itc limited, ITC Limited]",val
3,24581,Where is the university located ?,[others],<PIL.PngImagePlugin.PngImageFile image mode=L ...,7027,nkbl0226,1,"[san diego, San Diego]",val
4,24582,To whom is the document sent?,"[handwritten, form]",<PIL.PngImagePlugin.PngImageFile image mode=L ...,7027,nkbl0226,1,[Paul],val


In [23]:
def build_docvqa_page_key(record: Dict[str, Any]) -> str:
    document_id = normalize_text(record.get("ucsf_document_id", ""))
    page_no = normalize_text(record.get("ucsf_document_page_no", ""))

    if document_id and page_no:
        return f"{document_id}::page-{page_no}"

    doc_id = normalize_text(record.get("docId", ""))
    if doc_id:
        return f"doc-{doc_id}"

    return stable_id(DatasetName.DOCVQA.value, record.get("questionId"), record.get("question"))


def extract_docvqa_ocr_lines(image: Any, min_confidence: float = 35.0) -> List[str]:
    """Extract OCR lines from a DocVQA page image when Tesseract is available."""
    if not DOCVQA_ENABLE_OCR or pytesseract is None or image is None:
        return []

    if not globals().get("TESSERACT_READY", False):
        return []

    try:
        ocr_data = pytesseract.image_to_data(
            image.convert("RGB"),
            output_type=pytesseract.Output.DICT,
        )
    except Exception as exc:
        print(f"DocVQA OCR unavailable for one page: {exc}")
        return []

    grouped_tokens: Dict[Tuple[int, int, int], List[Dict[str, Any]]] = {}
    token_count = len(ocr_data.get("text", []))

    for index in range(token_count):
        token = normalize_text(ocr_data["text"][index])
        if not token:
            continue

        try:
            confidence = float(ocr_data["conf"][index])
        except (TypeError, ValueError):
            confidence = -1.0

        if confidence >= 0 and confidence < min_confidence:
            continue

        key = (
            int(ocr_data["block_num"][index]),
            int(ocr_data["par_num"][index]),
            int(ocr_data["line_num"][index]),
        )
        grouped_tokens.setdefault(key, []).append(
            {
                "text": token,
                "left": int(ocr_data["left"][index]),
                "top": int(ocr_data["top"][index]),
                "width": int(ocr_data["width"][index]),
                "height": int(ocr_data["height"][index]),
            }
        )

    ocr_lines: List[str] = []
    for line_index, (_key, tokens) in enumerate(sorted(grouped_tokens.items())):
        tokens = sorted(tokens, key=lambda token: token["left"])
        text = normalize_text(" ".join(token["text"] for token in tokens))
        if not text:
            continue

        left = min(token["left"] for token in tokens)
        top = min(token["top"] for token in tokens)
        right = max(token["left"] + token["width"] for token in tokens)
        bottom = max(token["top"] + token["height"] for token in tokens)
        ocr_lines.append(f"OCR Line {line_index} [{left},{top},{right},{bottom}]: {text}")

    return ocr_lines


def build_docvqa_page_text(record: Dict[str, Any]) -> str:
    ocr_lines = extract_docvqa_ocr_lines(record.get("image"))
    if ocr_lines:
        return normalize_text("\n".join(ocr_lines))

    document_id = normalize_text(record.get("ucsf_document_id", "unknown"))
    page_no = normalize_text(record.get("ucsf_document_page_no", "unknown"))
    return normalize_text(
        "\n".join(
            [
                f"OCR Line 0: OCR text unavailable for document {document_id}, page {page_no}.",
                "OCR Line 1: Install and configure Tesseract or a document OCR model to extract page text.",
            ]
        )
    )

In [24]:
def docvqa_group_to_document(page_key: str, group: pd.DataFrame) -> ArchiveDocument:
    records = group.to_dict(orient="records")
    first_record = records[0]

    raw_text = build_docvqa_page_text(first_record)
    source_id = normalize_text(page_key)
    document_id = stable_id(DatasetName.DOCVQA.value, source_id)

    question_ids = [normalize_text(record.get("questionId", "")) for record in records]
    question_types = sorted(
        {
            normalize_text(question_type)
            for record in records
            for question_type in (record.get("question_types") or [])
            if normalize_text(question_type)
        }
    )

    image = first_record.get("image")
    image_size = getattr(image, "size", None)

    return ArchiveDocument(
        document_id=document_id,
        source_id=source_id,
        dataset=DatasetName.DOCVQA.value,
        modality=Modality.OCR_DOCUMENT.value,
        title=f"DocVQA Page {source_id}",
        raw_text=raw_text,
        summary=normalize_text(f"DocVQA page with question types: {', '.join(question_types)}"),
        source_metadata={
            "hf_dataset": DOCVQA_DATASET_NAME,
            "hf_config": DOCVQA_CONFIG,
            "hf_split": DOCVQA_SPLIT,
            "doc_id": first_record.get("docId"),
            "ucsf_document_id": first_record.get("ucsf_document_id"),
            "ucsf_document_page_no": first_record.get("ucsf_document_page_no"),
            "question_ids": question_ids,
            "question_count": len(question_ids),
            "question_types": question_types,
            "data_split": first_record.get("data_split"),
            "image_size": image_size,
            "answers_excluded_from_embedding_metadata": True,
            "original_fields": [field for field in list(group.columns) if field != "answers"],
        },
    )


def docvqa_document_to_archive_chunks(
    document: ArchiveDocument,
    max_words: int = 250,
    overlap_words: int = 40,
) -> List[ArchiveChunk]:
    raw_chunks = chunk_text_by_words(
        document.raw_text,
        max_words=max_words,
        overlap_words=overlap_words,
    )

    chunks: List[ArchiveChunk] = []

    for chunk_index, raw_chunk in enumerate(raw_chunks):
        masked_text, sensitive_entities, sensitivity_level, access_level = apply_security_masking_placeholder(raw_chunk)
        chunk_id = stable_id(document.document_id, chunk_index, raw_chunk[:200])

        embedding_text = build_embedding_text(
            dataset=document.dataset,
            modality=document.modality,
            title=document.title,
            summary=document.summary,
            masked_text=masked_text,
            source_metadata=document.source_metadata,
        )

        chunks.append(
            ArchiveChunk(
                chunk_id=chunk_id,
                document_id=document.document_id,
                source_id=document.source_id,
                dataset=document.dataset,
                modality=document.modality,
                chunk_index=chunk_index,
                title=document.title,
                raw_text=raw_chunk,
                masked_text=masked_text,
                embedding_text=embedding_text,
                summary=document.summary,
                sensitivity_level=sensitivity_level,
                sensitive_entities=sensitive_entities,
                access_level=access_level,
                source_metadata=document.source_metadata,
            )
        )

    return chunks

In [25]:
docvqa_df = docvqa_df.copy()
docvqa_df["page_key"] = docvqa_df.apply(lambda row: build_docvqa_page_key(row.to_dict()), axis=1)

docvqa_documents = [
    docvqa_group_to_document(page_key, group)
    for page_key, group in docvqa_df.groupby("page_key", sort=True)
]

docvqa_chunks: List[ArchiveChunk] = []
for document in tqdm(docvqa_documents, desc="Chunking DocVQA OCR documents"):
    docvqa_chunks.extend(docvqa_document_to_archive_chunks(document))

docvqa_document_df = archive_documents_to_dataframe(docvqa_documents)
docvqa_chunk_df = archive_chunks_to_dataframe(docvqa_chunks)
docvqa_document_csv_path, docvqa_chunk_csv_path = dataset_cache_paths("docvqa")
dataframe_to_csv(docvqa_document_df, docvqa_document_csv_path, json_columns=["source_metadata"])
dataframe_to_csv(docvqa_chunk_df, docvqa_chunk_csv_path, json_columns=CSV_JSON_COLUMNS)

print(f"Page documents: {len(docvqa_documents):,}")
print(f"Chunks: {len(docvqa_chunks):,}")


Chunking DocVQA OCR documents:   0%|          | 0/1286 [00:00<?, ?it/s]

Saved 1,286 rows to data\csv_cache\docvqa_documents.csv
Saved 2,359 rows to data\csv_cache\docvqa_chunks.csv
Page documents: 1,286
Chunks: 2,359


In [ ]:
if "_load_archive_chunks_from_csv" not in globals():
    def _chunk_dataframe_to_archive_chunks(chunk_df: pd.DataFrame) -> List[ArchiveChunk]:
        chunk_df = chunk_df.copy()

        if "chunk_index" in chunk_df.columns:
            chunk_df["chunk_index"] = pd.to_numeric(chunk_df["chunk_index"], errors="coerce").fillna(0).astype(int)

        for column in ["dense_vector_ready", "sparse_vector_ready"]:
            if column in chunk_df.columns:
                chunk_df[column] = chunk_df[column].map(
                    lambda value: str(value).strip().lower() == "true"
                    if isinstance(value, str)
                    else bool(value)
                )

        chunk_df = chunk_df.where(pd.notna(chunk_df), None)
        return [ArchiveChunk(**record) for record in chunk_df.to_dict(orient="records")]


    def _read_chunk_csv_dataframe(
        csv_path: os.PathLike[str] | str,
        max_bytes: Optional[int] = None,
        chunksize: int = 50_000,
    ) -> pd.DataFrame:
        if max_bytes is None:
            return csv_to_dataframe(csv_path, json_columns=CSV_JSON_COLUMNS)

        frames = []
        loaded_bytes = 0

        for chunk_df in pd.read_csv(csv_path, chunksize=chunksize):
            chunk_bytes = int(chunk_df.memory_usage(index=True, deep=True).sum())
            remaining_bytes = max_bytes - loaded_bytes

            if chunk_bytes > remaining_bytes:
                average_row_bytes = max(chunk_bytes / max(len(chunk_df), 1), 1)
                rows_to_keep = max(int(remaining_bytes // average_row_bytes), 0)
                if rows_to_keep > 0:
                    chunk_df = chunk_df.iloc[:rows_to_keep].copy()
                    frames.append(chunk_df)
                    loaded_bytes += int(chunk_df.memory_usage(index=True, deep=True).sum())
                break

            frames.append(chunk_df)
            loaded_bytes += chunk_bytes

            if loaded_bytes >= max_bytes:
                break

        if not frames:
            return pd.DataFrame()

        chunk_df = pd.concat(frames, ignore_index=True)
        for column in CSV_JSON_COLUMNS:
            if column in chunk_df.columns:
                chunk_df[column] = chunk_df[column].map(_json_parse_cell)

        print(f"Loaded approximately {loaded_bytes / (1024 ** 3):.2f} GiB from {csv_path}")
        return chunk_df


    def _load_archive_chunks_from_csv(
        csv_path: os.PathLike[str] | str,
        max_bytes: Optional[int] = None,
    ) -> Tuple[pd.DataFrame, List[ArchiveChunk]]:
        chunk_df = _read_chunk_csv_dataframe(csv_path, max_bytes=max_bytes)
        chunks = _chunk_dataframe_to_archive_chunks(chunk_df)
        print(f"Loaded chunks from CSV: {len(chunks):,}")
        return chunk_df, chunks


CHUNK_CSV_READ_LIMIT_BYTES = 1 * 1024 ** 3

_, docvqa_chunk_csv_path = dataset_cache_paths("docvqa")
docvqa_chunk_df, docvqa_chunks = _load_archive_chunks_from_csv(docvqa_chunk_csv_path)


In [26]:
preview_columns = [
    "chunk_id",
    "document_id",
    "dataset",
    "modality",
    "chunk_index",
    "sensitivity_level",
    "access_level",
    "raw_text",
    "masked_text",
]

docvqa_chunk_df[preview_columns].head()


,chunk_id,document_id,dataset,modality,chunk_index,sensitivity_level,access_level,raw_text,masked_text
0,19d830bf12bb56e08f15effd,9da36463be3bd89838c34511,docvqa,ocr_document,0,S0,public,"OCR Line 0 [296,256,639,281]: Robert E. Shank,...","OCR Line 0 [296,256,639,281]: Robert E. Shank,..."
1,b731c45696f6800cedda1d09,ad0a89790442222adff356d5,docvqa,ocr_document,0,S0,public,"OCR Line 0 [189,92,961,118]: VANDERBILT UNIVER...","OCR Line 0 [189,92,961,118]: VANDERBILT UNIVER..."
2,a3d1a39fd2629b17ba950692,b2ca1d3d986a3d35bf70aaeb,docvqa,ocr_document,0,S0,public,"OCR Line 0 [491,301,1271,326]: ALCOHOLIC BEVER...","OCR Line 0 [491,301,1271,326]: ALCOHOLIC BEVER..."
3,c0f249fc173ac432fddb36c4,b2ca1d3d986a3d35bf70aaeb,docvqa,ocr_document,1,S0,public,"OCR Line 29 [105,1983,1052,2013]: *Reclassifie...","OCR Line 29 [105,1983,1052,2013]: *Reclassifie..."
4,7614e84e2c58a57ee0ddeeb1,3afd62238f88f379354f8a87,docvqa,ocr_document,0,S0,public,"OCR Line 0 [99,23,1213,62]: SENT REYNOLDS TOB....","OCR Line 0 [99,23,1213,62]: SENT REYNOLDS TOB...."


In [27]:
# Inspect one future embedding input.
# This should use masked_text, not raw_text, once the masking layer is implemented.
print(docvqa_chunks[0].embedding_text[:2_000])

Dataset: docvqa
Modality: ocr_document
Title: DocVQA Page ffbg0227::page-2
Summary: DocVQA page with question types: layout
Metadata: {"hf_dataset": "lmms-lab/DocVQA", "hf_config": "DocVQA", "hf_split": "validation", "doc_id": 7467, "ucsf_document_id": "ffbg0227", "ucsf_document_page_no": "2", "question_ids": ["26167"], "question_count": 1, "question_types": ["layout"], "data_split": "val", "image_size": [1786, 2285], "answers_excluded_from_embedding_metadata": true, "original_fields": ["questionId", "question", "question_types", "image", "docId", "ucsf_document_id", "ucsf_document_page_no", "data_split", "page_key"]}
Content: OCR Line 0 [296,256,639,281]: Robert E. Shank, M.D.
OCR Line 1 [297,291,393,316]: Page 2
OCR Line 2 [295,323,526,350]: April 17, 1972
OCR Line 3 [457,419,1526,448]: As you know, each candidate selects the area of special emphasis
OCR Line 4 [293,452,1509,480]: to be covered in depth in his examinations. We are attempting to compile
OCR Line 5 [293,486,1490,514]: 

## 3. Embedding and Qdrant Indexing

This section embeds the normalized archive chunks and inserts them into Qdrant. We use `BAAI/bge-m3` through the reusable `BGEM3Embedder` class in `src/bge_m3_embedding.py`.

BGE-M3 is useful here because one model produces both:

- `dense`: semantic dense vectors for neural retrieval.
- `sparse`: lexical sparse vectors for sparse or hybrid dense-sparse retrieval.

The inserted Qdrant points keep the full chunk payload for traceability, but vectors are generated from `embedding_text`, which is built from `masked_text` plus useful metadata context.

In [28]:
QDRANT_DENSE_VECTOR_NAME = "dense"
QDRANT_SPARSE_VECTOR_NAME = "sparse"
QDRANT_DISTANCE = models.Distance.COSINE

# BGE-M3 supports up to 8192 tokens, but that is slow on CPU.
# 1024 is a good prototype default for chunk-sized archive text.
EMBEDDING_MAX_LENGTH = 1024
EMBEDDING_BATCH_SIZE = 8
QDRANT_UPSERT_BATCH_SIZE = 32
SKIP_EXISTING_QDRANT_POINTS = True
MAX_CHUNKS_TO_INDEX: Optional[int] = None  # Use None when you are ready for a full indexing run.


def ensure_archive_chunk_collection(
    client: QdrantClient,
    collection_name: str = COLLECTION_NAME,
) -> None:
    """Create the archive chunk collection if it does not already exist."""
    if client.collection_exists(collection_name=collection_name):
        print(f"Qdrant collection already exists: {collection_name}")
        return

    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            QDRANT_DENSE_VECTOR_NAME: models.VectorParams(
                size=BGE_M3_DENSE_VECTOR_SIZE,
                distance=QDRANT_DISTANCE,
            ),
        },
        sparse_vectors_config={
            QDRANT_SPARSE_VECTOR_NAME: models.SparseVectorParams(),
        },
    )
    print(f"Created Qdrant collection: {collection_name}")

In [ ]:
chunk_variable_names = [
    "mediasum_chunks",
    "cnn_dailymail_chunks",
    "msr_vtt_chunks",
    "docvqa_chunks",
]

all_archive_chunks: List[ArchiveChunk] = []
for variable_name in chunk_variable_names:
    chunks = globals().get(variable_name, [])
    print(f"{variable_name}: {len(chunks):,}")
    all_archive_chunks.extend(chunks)

if MAX_CHUNKS_TO_INDEX is not None:
    chunks_to_index = all_archive_chunks[:MAX_CHUNKS_TO_INDEX]
else:
    chunks_to_index = all_archive_chunks

print(f"Total chunks available: {len(all_archive_chunks):,}")
print(f"Chunks selected for indexing: {len(chunks_to_index):,}")

cnn_dailymail_chunks: 930,585
msr_vtt_chunks: 6,513
docvqa_chunks: 2,359
Total chunks available: 939,457
Chunks selected for indexing: 939,457


In [ ]:
bge_m3_embedder = BGEM3Embedder(
    model_name=BGE_M3_MODEL_NAME,
    use_fp16=True,
)

print(f"Loaded embedding model: {BGE_M3_MODEL_NAME}")

In [ ]:
def existing_qdrant_point_ids(
    client: QdrantClient,
    collection_name: str,
    point_ids: List[str],
) -> set:
    """Return point IDs that already exist in Qdrant."""
    if not point_ids:
        return set()

    existing = client.retrieve(
        collection_name=collection_name,
        ids=point_ids,
        with_payload=False,
        with_vectors=False,
    )
    return {str(point.id) for point in existing}


def embed_and_upsert_chunks_to_qdrant(
    chunks: List[ArchiveChunk],
    client: QdrantClient,
    collection_name: str = COLLECTION_NAME,
    embedding_batch_size: int = EMBEDDING_BATCH_SIZE,
    upsert_batch_size: int = QDRANT_UPSERT_BATCH_SIZE,
    max_length: int = EMBEDDING_MAX_LENGTH,
    skip_existing: bool = SKIP_EXISTING_QDRANT_POINTS,
) -> int:
    """Embed chunks with BGE-M3 and upsert dense+sparse vectors to Qdrant."""
    if not chunks:
        print("No chunks to index.")
        return 0

    ensure_archive_chunk_collection(client, collection_name)

    if skip_existing:
        filtered_chunks: List[ArchiveChunk] = []
        for chunk_batch in batched(chunks, 256):
            point_ids = qdrant_point_ids(chunk_batch)
            existing_ids = existing_qdrant_point_ids(client, collection_name, point_ids)
            filtered_chunks.extend(
                chunk
                for chunk, point_id in zip(chunk_batch, point_ids)
                if point_id not in existing_ids
            )

        skipped_count = len(chunks) - len(filtered_chunks)
        print(f"Skipping chunks already indexed in Qdrant: {skipped_count:,}")
        chunks = filtered_chunks

    if not chunks:
        print("All selected chunks are already indexed.")
        return 0

    total_indexed = 0
    pending_points = []
    chunk_batches = list(batched(chunks, embedding_batch_size))

    for chunk_batch in tqdm(chunk_batches, desc="Embedding archive chunks with BGE-M3"):
        embeddings = bge_m3_embedder.encode_chunks(
            chunk_batch,
            batch_size=embedding_batch_size,
            max_length=max_length,
        )
        pending_points.extend(build_qdrant_points(chunk_batch, embeddings))

        while len(pending_points) >= upsert_batch_size:
            points_to_upsert = pending_points[:upsert_batch_size]
            pending_points = pending_points[upsert_batch_size:]
            client.upsert(
                collection_name=collection_name,
                points=points_to_upsert,
                wait=True,
            )
            total_indexed += len(points_to_upsert)

    if pending_points:
        client.upsert(
            collection_name=collection_name,
            points=pending_points,
            wait=True,
        )
        total_indexed += len(pending_points)

    print(f"Indexed chunks into Qdrant: {total_indexed:,}")
    return total_indexed

In [ ]:
indexed_chunk_count = embed_and_upsert_chunks_to_qdrant(
    chunks=chunks_to_index,
    client=qdrant_client,
    collection_name=COLLECTION_NAME,
    max_length=EMBEDDING_MAX_LENGTH,
)

indexed_chunk_count

Qdrant collection already exists: archive-chunks
Skipping chunks already indexed in Qdrant: 2


Embedding archive chunks with BGE-M3:   0%|          | 0/13 [00:00<?, ?it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00,  7.78it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings:   0%|          | 0/1 [00:21<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
qdrant_client.get_collection(COLLECTION_NAME)